# Generative Backend — Pick-Up Pipeline

Demonstrates the end-to-end NL → PickUpAction pipeline using a real SDT world.

**Pipeline stages shown step by step:**
1. **Slot filling** — LLM extracts a `PickUpSlotSchema` from the instruction
2. **Entity grounding** — maps the object description to a world `Body`
3. **PartialDesignator** — combines slot schema + grounded body, shows free variables
4. **Hybrid resolution** — LLM fills the remaining discrete parameters
5. **Full run** — `pipeline.run()` executes all stages in one call

**Prerequisites:** `cd cognitive_robot_abstract_machine && uv sync --active`  
**API key:** set `OPENAI_API_KEY` in `generative_backend/.env`

## 1 · Environment & Imports

In [ ]:
import logging
import pathlib
from dotenv import load_dotenv

# Load API keys before any LLM import
_env = pathlib.Path().resolve().parent / ".env"
load_dotenv(_env, override=True)

logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")

# ── SDT / pycram stack ────────────────────────────────────────────────────────
from semantic_digital_twin.robots.pr2 import PR2
from pycram.testing import setup_world

# ── Generative backend ────────────────────────────────────────────────────────
from generative_backend.pipeline import PickUpPipeline

print("All imports OK")

## 2 · Load World

`setup_world()` loads the apartment URDF, a PR2 robot, a milk carton (`milk.stl`), and a breakfast cereal box.
The milk carton sits on the kitchen counter at roughly `(2.37, 2.0, 1.05)`.

In [ ]:
world = setup_world()

try:
    robot = PR2.from_world(world)
except Exception as exc:
    # Partial setup if the full collision matrix cannot be built
    print(f"Note: PR2 full setup skipped ({type(exc).__name__}) — recovering annotation")
    robot = next((a for a in world.semantic_annotations if isinstance(a, PR2)), None)
    if robot is None:
        raise RuntimeError("Could not obtain PR2 annotation") from exc
    with world.modify_world():
        robot._setup_velocity_limits()
        robot._setup_hardware_interfaces()
        robot._setup_joint_states()

# Confirm the milk body is present
milk_body = world.get_body_by_name("milk.stl")
pos = milk_body.global_pose.to_position()  # HomogeneousTransformationMatrix → Point3
print(f"World loaded — milk at x={float(pos.x):.2f}, y={float(pos.y):.2f}, z={float(pos.z):.2f}")

## 3 · Initialise Pipeline

`PickUpPipeline.from_world()` auto-detects the robot's manipulator from semantic annotations.

In [ ]:
pipeline = PickUpPipeline.from_world(world)

print(f"Pipeline ready")
print(f"  manipulator : {type(pipeline.world_context.manipulator).__name__}")
print(f"  eql_preferred: {pipeline.eql_preferred}")

## 4 · Stage 1a — Slot Filling

The LLM receives the instruction and returns a `PickUpSlotSchema`.  
Fields not mentioned in the instruction are left `null` (free variables for Phase 2).

In [ ]:
INSTRUCTION = "Pick up the milk from the counter"

slot_schema = pipeline.extract_slots(INSTRUCTION)

print(f"Instruction : {INSTRUCTION!r}")
print()
print("PickUpSlotSchema:")
print(f"  action_type       : {slot_schema.action_type}")
print(f"  object.name       : {slot_schema.object_description.name}")
print(f"  object.sem_type   : {slot_schema.object_description.semantic_type}")
print(f"  object.spatial    : {slot_schema.object_description.spatial_context}")
print(f"  object.attributes : {slot_schema.object_description.attributes}")
print(f"  arm               : {slot_schema.arm}   ← null means unspecified")
print(f"  grasp_params      : {slot_schema.grasp_params}   ← null means unspecified")

## 5 · Stage 1b — Entity Grounding

The object description is matched against world bodies.  
Tier 1 uses substring + semantic-type filtering; Tier 2 is a pure fuzzy substring fallback.

In [ ]:
grounding_result = pipeline.ground(slot_schema)

body_names = [
    str(getattr(getattr(b, "name", None), "name", b))
    for b in grounding_result.bodies
]

print(f"Bodies found : {body_names}")
print(f"Used EQL     : {grounding_result.used_eql}")
print(f"Warning      : {grounding_result.warning or 'none'}")

## 6 · Stage 1c — PartialDesignator

Combines the slot schema and grounded body into a `PartialDesignator[PickUpAction]`.  
Parameters not yet resolved appear as `None` (free variables).

In [ ]:
partial = pipeline.build_partial(slot_schema, grounding_result)

print("PartialDesignator[PickUpAction]:")
for key, val in partial.kwargs.items():
    status = "✓ filled" if val is not None else "✗ free (needs Phase 2)"
    print(f"  {key:25s}: {val!r}   {status}")

missing = partial.missing_parameter()
print()
print(f"Missing parameters: {missing if missing else 'none — fully specified'}")

## 7 · Stage 2 — Hybrid Resolution

The LLM receives a world context snapshot (object pose relative to robot, semantic annotations)  
and resolves the remaining discrete parameters: `arm`, `approach_direction`, `vertical_alignment`, `rotate_gripper`.

In [ ]:
from generative_backend.hybrid_resolver import HybridPickUpResolver, WorldContextBuilder

# Show what the LLM sees
ctx_str = WorldContextBuilder(world).build_for_pickup(partial)
print("World context sent to resolver LLM:")
print(ctx_str)
print()

In [ ]:
action = pipeline.resolve(partial)

gd = action.grasp_description
print("Resolved PickUpAction:")
print(f"  object_designator : {getattr(getattr(action.object_designator, 'name', None), 'name', action.object_designator)}")
print(f"  arm               : {action.arm}")
print(f"  approach_direction: {gd.approach_direction}")
print(f"  vertical_alignment: {gd.vertical_alignment}")
print(f"  rotate_gripper    : {gd.rotate_gripper}")
print(f"  manipulator       : {type(gd.manipulator).__name__}")

## 8 · Full Pipeline Run

`pipeline.run()` executes all stages above in one call.

In [ ]:
action = pipeline.run(INSTRUCTION)

gd = action.grasp_description
print(f"PickUpAction from pipeline.run({INSTRUCTION!r}):")
print(f"  object : {getattr(getattr(action.object_designator, 'name', None), 'name', action.object_designator)}")
print(f"  arm    : {action.arm}")
print(f"  approach_direction : {gd.approach_direction}")
print(f"  vertical_alignment : {gd.vertical_alignment}")
print(f"  rotate_gripper     : {gd.rotate_gripper}")